# Frozen GPT-2 text-only reference → ZuCo sentiment

This notebook measures three-way sentiment performance when the true stimulus sentence is supplied to a frozen standard GPT-2 model. It uses no EEG and no reader rows. It is a separate reference, not V5 and not part of the V1-V4 EEG stoplight family.

Select a **GPU** runtime and run every cell in order. The first run downloads about 552 MB of pinned GPT-2 weights/tokenizer files to Google Drive; later runs reuse them.

In [ ]:
# 1) Fetch this project and verify Colab's already-installed runtime packages.
from pathlib import Path
import importlib.metadata as package_metadata
import os, subprocess, sys

PROJECT_URL = "https://github.com/parmisbathayan/EEGTokenizer.git"
PROJECT_ROOT = Path("/content/EEGTokenizer")

def run(command, **kwargs):
    print("+", " ".join(map(str, command)))
    return subprocess.run(command, check=True, text=True, **kwargs)

if not PROJECT_ROOT.exists():
    run(["git", "clone", "--depth", "1", PROJECT_URL, str(PROJECT_ROOT)])
else:
    run(["git", "pull", "--ff-only"], cwd=PROJECT_ROOT)
os.chdir(PROJECT_ROOT / "neurolm")
run([sys.executable, "-c", "import torch, transformers, sklearn, pandas, numpy; print('runtime import check passed')"])
run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-q"], cwd=Path.cwd())
print("torch:", package_metadata.version("torch"))
print("transformers:", package_metadata.version("transformers"))
print("scikit-learn:", package_metadata.version("scikit-learn"))

In [ ]:
# 2) Mount Drive and edit only these paths if your thesis layout differs.
from google.colab import drive
drive.mount("/content/drive")

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis")
LABELS_CSV = THESIS_ROOT / "Data/zuco_sentiment_labels_task1_fixed.csv"
CACHE_ROOT = THESIS_ROOT / "CachedArtifacts/eeg_tokenizer/neurolm"
MODEL_CACHE = CACHE_ROOT / "text_gpt2_hf_cache"
FEATURE_CACHE = CACHE_ROOT / "text_gpt2_features_v1.npz"
RESULTS_DIR = THESIS_ROOT / "Results/eeg_tokenizer/neurolm/text_gpt2_reference"

if not LABELS_CSV.exists():
    raise FileNotFoundError(f"Missing fixed label table: {LABELS_CSV}")
for path in (MODEL_CACHE, RESULTS_DIR):
    path.mkdir(parents=True, exist_ok=True)
print("Labels:", LABELS_CSV)
print("Persistent GPT-2 cache:", MODEL_CACHE)
print("Feature cache:", FEATURE_CACHE)
print("Results:", RESULTS_DIR)

In [ ]:
# 3) Extract/reuse frozen text features and run the locked nested CV.
import json
import torch
from src.text_gpt2 import TextGPT2Config, run_text_reference

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    print("WARNING: no GPU detected; extraction will run on CPU.")
config = TextGPT2Config()
metrics, predictions, summary, delta = run_text_reference(
    labels_csv=LABELS_CSV,
    feature_cache=FEATURE_CACHE,
    model_cache_dir=MODEL_CACHE,
    output_dir=RESULTS_DIR,
    device=DEVICE,
    config=config,
    force_extract=False,
)
display(summary)
print(json.dumps(delta, indent=2))

In [ ]:
# 4) Validate provenance and show the direct macro-F1 comparison.
import pandas as pd

extraction = json.loads((RESULTS_DIR / "extraction_report.json").read_text())
assert extraction["sentences"] == 400, extraction
assert extraction["truncated_sentences"] == 0, extraction
text_f1 = float(summary.loc[summary.setup == "gpt2_text_probe", "macro_f1_mean"].iloc[0])
comparison = pd.DataFrame([
    {"system": "GPT-2 sentence text", "input": "text", "macro_f1": text_f1},
    {"system": "V1 pooled NeuroLM", "input": "EEG", "macro_f1": 0.3493},
    {"system": "V2 raw EEGNet", "input": "EEG", "macro_f1": 0.3102},
    {"system": "V3 structured NeuroLM", "input": "EEG", "macro_f1": 0.2455},
]).sort_values("macro_f1", ascending=False)
display(comparison)
print(json.dumps(extraction, indent=2))
print("Saved result files:")
for path in sorted(RESULTS_DIR.iterdir()):
    print(" -", path.name, path.stat().st_size, "bytes")

## Interpretation

Use `gpt2_text_probe` macro-F1 as the text-only reference. The shuffled-pairing result checks that the score depends on the correct sentence/label association, and majority is the minimum baseline. A strong text score means the labels are recoverable from the sentences; it is not an expected ceiling for EEG. Do not tune the EEG versions based on this result.